# Fourth Notebook to use
Data only missing encoding which I will do in here instead, though realistically it matters little where it is done.

In [37]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import ast
from typing import List
from sklearn.preprocessing import MultiLabelBinarizer

First thing: encode categorical features

In [39]:
def cat_to_list(df: pd.DataFrame, columnnames: List[str]):
    for column in columnnames:
        if type(df.loc[0,column]) == str:
            df[column] = df[column].apply(lambda x: ast.literal_eval(x))
    return df
cat_features = ['genre', 'country', 'language']

In [40]:
df_data = pd.read_csv('initial_dframe_direc_stats.csv')
df_data = cat_to_list(df_data, cat_features)
df_unseen = pd.read_csv('unseen_movies_direc_stats.csv')
df_unseen = cat_to_list(df_unseen, cat_features)

In [142]:
def encode_category(dframe, column_name, name_of_new_column, rare_threshold = 1):
    mlb = MultiLabelBinarizer()
    column_encoded = mlb.fit_transform(dframe[column_name])
    # make a matrix with columns like
    # title     drama   crime   comedy  romance
    #   A       1       0       1       0
    #   B       1       0       0       0
    #   C       0       0       1       1
    # and so on. 
    new_dframe = pd.DataFrame(column_encoded, 
                        columns=[f"{column_name}: {g}" for g in mlb.classes_], index=dframe.index)
    # now that matrix turned into a df, with column names: genre: drama, genre: crime etc.
    
    # make this as a new column for rare categories.
    column_names = new_dframe.columns   
    new_dframe[name_of_new_column] = 0
    
    # the for loop below goes through each new column made above, and counts the total number of
    # entries. If it is lower than 2, I consider it rare, and then make sure the only row that has
    # a 1 in that column also gets a 1 in the new rare column. That original column is then deleted.
    for category in column_names:
        if new_dframe[category].sum() <= rare_threshold: # if rare category
            new_dframe[name_of_new_column] = new_dframe[name_of_new_column] | new_dframe[category] 
            # 1 in same row if 1 already in new column or 1 in category, else 0
            new_dframe = new_dframe.drop(category, axis=1) # drop the rare category
    
    dframe = dframe.drop(column_name, axis=1)

    return pd.concat([dframe, new_dframe], axis=1)

def encode_dataset(dframe: pd.DataFrame, columnnames: List[str], rare_threshold: int = 1):
    dframe_encoded = dframe[:]
    for column in columnnames:
        dframe_encoded = encode_category(dframe_encoded, column, f'rare {column}', rare_threshold=rare_threshold)
    return dframe_encoded

# this function below is the one used on a pair. One dataset to encode freely, the other to follow the encoding of the first

def encode_data(dframe_data: pd.DataFrame, dframe_unseen: pd.DataFrame, columns_to_encode: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]: 
    """
    Encode training/full dataset as well as the unseen movies dataset at the same time. Returns the encoded versions of the training/full dataset and the unseen movies dataset.
    """
    dfs_data = []
    dfs_unseen = []
    for column in columns_to_encode:
        rare_name = f'rare {column}'
        dframe_data_enc = encode_category(dframe_data, column, rare_name)
        dframe_unseen_enc = encode_category(dframe_unseen, column, rare_name, rare_threshold=0) # threshold=0 for nothing put in rare category
        cols_data = set(dframe_data_enc.columns)
        cols_unseen = set(dframe_unseen_enc.columns)
        cols_data_only = cols_data - cols_unseen
        cols_unseen_only = cols_unseen - cols_data
        for col in cols_data_only:                          # add missing cols from data to unseen
            dframe_unseen_enc[col] = 0

        # now to remove columns in unseen not in training and fill in rare instead
        dframe_unseen_enc[rare_name] = (dframe_unseen_enc[list(cols_unseen_only)].sum(axis=1) > 0).astype(int)
                                                # list for pandas     # sums row-wise, > 0 is True if any to-be-removed-column has a 1. Then astype(int) to turn true and false into 1 and 0
        dframe_unseen_enc.drop(columns=cols_unseen_only)
        dfs_data.append(dframe_data_enc)
        dfs_unseen.append(dframe_data_enc)
    dframe_data_enc = pd.concat(dfs_data)
    dframe_unseen_enc = pd.concat(dfs_unseen)
    dframe_data_enc.drop(columns=columns, inplace=True)
    dframe_unseen_enc.drop(columns=columns, inplace=True)

    return dframe_data_enc, dframe_unseen_enc

In [133]:
df_data_enc, df_unseen_enc = encode_data(df_data, df_unseen, cat_features)

I want to calculate the average of all my personal ratings ive given to each movie of a specific director, to have some way of meassure how much I like each director.

So I need to calculate a feature after splitting to training, validation and test sets. The standard way to do this such that it's even respected in a cv fold is through a Pipeline. 

Just like with using PolynomialFeatures in a pipeline, it's the same idea here! So first define an encoder.

In [2]:
class DirectorTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y):    # this is ONLY called during training!
        df = X.copy()
        df['_target'] = y

        self.global_mean = y.mean()

        stats = (
            df.melt(id_vars="_target", value_vars=["director1", "director2"])
            .dropna()
            .groupby("value")["_target"]
            .mean()
        )

        self.mapping_ = stats.to_dict()
        return self
    
    def transform(self, X): # this is called during training, validation and unseen data!
        avg_avg_personal_scores = []
        for d1, d2 in zip(X['director1'], X['director2']):
            s1 = self.mapping_.get(d1, self.global_mean)
            s2 = self.mapping_.get(d2, self.global_mean) if d2 else None

            avg_avg_personal_scores.append(s1 if not s2 else (s1 + s2)/2)
        
        return pd.DataFrame({'avg_personal_score': avg_avg_personal_scores}, index=X.index)

        

In [3]:
preprocess = ColumnTransformer(
    transformers=[
        ("director", DirectorTargetEncoder(), ["director1", "director2"]),
    ],
    remainder="passthrough" # all other columns are automatically included
) 

# Make training and unseen data correct format! So finalize presentation of datasets

In [4]:
df = pd.read_csv('initial_dframe_direc_stats.csv')
df.head(2)

,title,release_year,personal_rating,avg_rating,genre,country,language,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies,director1,director2
0,2 Fast 2 Furious,2003,2.5,3.16,"['Crime', 'Action', 'Thriller']","['Germany', 'USA']","['English', 'English', 'Spanish']",12.0,3.357333,3.0869,john singleton,NaN
1,2012,2009,2.0,2.56,"['Adventure', 'Action', 'Science Fiction']",['USA'],"['English', 'Russian', 'Hindi', 'German', 'Ita...",19.0,3.075132,3.1184,roland emmerich,NaN


In [5]:
df_unseen = pd.read_csv('unseen_movies_direc_stats.csv')
df_unseen.head(2)

,title,release_year,avg_rating,genre,country,language,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies,director1,director2
0,12 angry men,1957,4.2755,['Drama'],['US'],['English'],50.0,3.18903,3.2055,sidney lumet,NaN
1,20th century girl,2022,4.0725,"['Romance', 'Drama']",['KR'],['Korean'],1.0,4.07250,4.0725,bang woo-ri,NaN


Make all categorical values to actual list of strings first.

In [6]:
# need to convert the string of list of strings to just a list of strings to do encoding
cat_features = ['genre', 'country', 'language']

for column in cat_features:
    df[column] = df[column].apply(lambda x: ast.literal_eval(x))
    df_unseen[column] = df_unseen[column].apply(lambda x: ast.literal_eval(x))

In [ ]:
# from useful_funcs import normalize_country
# df['country'] = df['country'].apply(normalize_country)

In [8]:
df.melt(id_vars="personal_rating", value_vars=["director1", "director2"])

,personal_rating,variable,value
0,2.5,director1,john singleton
1,2.0,director1,roland emmerich
2,0.5,director1,sam mendes
3,3.5,director1,mary harron
4,4.0,director1,thomas vinterberg
...,...,...,...
229,2.5,director2,NaN
230,3.5,director2,NaN
231,3.5,director2,NaN
232,3.5,director2,NaN


In [52]:
for director in df['director1']:
    pass

In [9]:
unique_directors = pd.concat([df["director1"], df["director2"]]).dropna().unique()
unique_directors = [d for d in unique_directors if d != np.nan]

In [10]:
avg_personal_scores = []
nums = []
for director in unique_directors:
    mean = df[(df['director1'] == director) | (df['director2'] == director)]['personal_rating'].mean()
    num = len(df[(df['director1'] == director) | (df['director2'] == director)])
    avg_personal_scores.append(mean)
    nums.append(num)

In [11]:
df2 = pd.DataFrame({
    'avg_personal_score': avg_personal_scores
},
index=unique_directors)

In [19]:
pd.isna(df.loc[0, 'director2'])

True

In [20]:
avg_avg_personal_scores = []
for d1, d2 in zip(df['director1'], df['director2']):
        if pd.isna(d2):
                avg_avg_personal_scores.append(df2['avg_personal_score'].get(d1))
        else:
                avg_avg_personal_scores.append((df2['avg_personal_score'].get(d1)+df2['avg_personal_score'].get(d2))/2)
        

In [21]:
df[(df['director1'] == 'joe russo') | (df['director2'] == 'joe russo')]['personal_rating'].mean()

np.float64(4.0)

In [22]:
df2 = df.groupby('director1')['personal_rating'].agg(mean_rating = 'mean', num = 'count').reset_index()
df2.sort_values('num', ascending=False)

,director1,mean_rating,num
11,christopher nolan,3.900000,10
25,james cameron,2.833333,6
9,chris columbus,2.000000,4
50,peter jackson,2.250000,4
44,michael bay,2.750000,4
...,...,...,...
67,thomas vinterberg,4.000000,1
68,thorbjørn christoffersen,2.500000,1
69,tim burton,2.000000,1
70,tim miller,3.500000,1


In [27]:
mean_map = df2.set_index('director1')['mean_rating']
df['avg_personal_director_rating'] = df['director1'].map(mean_map)
df.sort_values('avg_personal_director_rating', ascending=False).head(10)

,title,release_year,personal_rating,avg_rating,genre,country,language,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies,director1,director2,avg_personal_director_rating
66,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"[Animation, Science Fiction, Adventure, Action]",[USA],"[English, English, Hindi, Italian, Spanish]",1.666667,4.129333,4.129333,kemp powers,justin k. thompson,5.0
91,The Intouchables,2011,5.0,4.15,"[Drama, Comedy]",[France],"[French, English, French]",10.000000,3.320400,3.535200,olivier nakache,éric toledano,5.0
102,The Shawshank Redemption,1994,5.0,4.58,"[Crime, Drama]",[USA],[English],7.000000,3.607571,3.850600,frank darabont,NaN,5.0
67,Spider-Man: Into the Spider-Verse,2018,4.5,4.40,"[Adventure, Animation, Action, Science Fiction]",[USA],"[English, English, Japanese, Spanish]",3.000000,3.768050,3.768050,bob persichetti,peter ramsey,4.5
50,KPop Demon Hunters,2025,4.5,3.58,"[Fantasy, Comedy, Music, Animation]",[USA],"[English, English, Korean]",1.500000,4.032375,4.032375,chris appelhans,maggie kang,4.5
116,Whiplash,2014,4.5,4.42,"[Drama, Music]",[USA],[English],7.000000,3.601500,3.717100,damien chazelle,NaN,4.5
33,Harry Potter and the Deathly Hallows: Part 1,2010,4.5,3.73,"[Adventure, Fantasy]","[UK, USA]",[English],13.000000,3.458462,3.331800,david yates,NaN,4.0
49,Jujutsu Kaisen 0,2021,4.0,3.92,"[Action, Animation, Fantasy]",[Japan],[Japanese],2.000000,3.818000,3.818000,sunghoo park,NaN,4.0
34,Harry Potter and the Deathly Hallows: Part 2,2011,4.0,4.01,"[Fantasy, Adventure]","[UK, USA]",[English],13.000000,3.458462,3.331800,david yates,NaN,4.0
9,Avengers: Endgame,2019,3.0,3.94,"[Science Fiction, Action, Adventure]",[USA],"[English, English, Japanese, Xhosa]",10.000000,3.351100,3.721200,joe russo,anthony russo,4.0


In [30]:
df[df['director2'] == 'anthony russo']

,title,release_year,personal_rating,avg_rating,genre,country,language,director_avg_num_movies,director_avg_movie_score,director_avg_last_5_movies,director1,director2,avg_personal_director_rating
9,Avengers: Endgame,2019,3.0,3.94,"[Science Fiction, Action, Adventure]",[USA],"[English, English, Japanese, Xhosa]",10.0,3.3511,3.7212,joe russo,anthony russo,4.0
10,Avengers: Infinity War,2018,5.0,4.02,"[Action, Science Fiction, Adventure]",[USA],"[English, English, Xhosa]",10.0,3.3511,3.7212,joe russo,anthony russo,4.0
